In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, classification_report
import joblib
import warnings

In [ ]:

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")


## Load & Save the Dataset


In [ ]:
!wget https://raw.githubusercontent.com/ahmed123234/Smart-Farm---Crop-Recommendation-System/refs/heads/main/Crop_recommendation.csv

In [ ]:
# --- 1. Data Loading and Setup ---
FILE_PATH = 'Crop_recommendation.csv'
MODEL_FILE = 'best_model.pkl'
SCALER_FILE = 'scaler.pkl'

try:
    df = pd.read_csv(FILE_PATH)
    print(f"Data loaded successfully from {FILE_PATH}. Total records: {len(df)}")
except FileNotFoundError:
    print(f"Error: {FILE_PATH} not found. Please ensure the Kaggle dataset is in the same directory.")
    exit()


## Data preperation and  EDA

### Task:
Prepare the data and make EDA analysis regarding value ranges, missing values, and target variable distribution.


### EDA includes:

1.  **Handle the Missing Values**

2.  **Find Target Variable (Crop) Distribution**
3.  **Feature Statistical Summary**

In [ ]:
# Data preparation and data cleaning
print("\n--- 2.1. Initial Data Overview ---")
print(df.head())
print("-" * 30)

print("\n--- 2.2. Checking for Missing Values ---")
print(df.isnull().sum())
print("-" * 30)
if df.isnull().sum().any():
    print("ACTION REQUIRED: Missing values must be addressed (e.g., imputation).")
    df.fillna(0, inplace=True)
    print("✅ Missing values replaced with 0.")
else:
    print("✅ No missing values found. Data quality is high.")

print("\n--- 2.3. Fix Data Types ---")
print(df.dtypes)
print("-" * 30)

print("\n--- 2.4 Identify and remove exact duplicate rows that represent the same entity ---")
df.drop_duplicates(inplace=True)
print("Duplicate rows are equal to {}".format(df.duplicated().sum()))
print("-" * 30)


In [ ]:
# --- 2. Exploratory Data Analysis (EDA) and Feature Analysis ---

print("\n--- 2.5. Target Variable (Crop) Distribution ---")
# Check how many samples we have for each crop
print(df['label'].value_counts())
print("-" * 30)
# Insight: A balanced dataset is key for classification.
if df['label'].value_counts().std() < 5: # Small threshold for standard deviation
    print("✅ Dataset is perfectly balanced (100 records per crop), ideal for training.")
else:
    print("CAUTION: Dataset is imbalanced. May require techniques like oversampling/undersampling.")

# Feature set definition
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = df[features]
y = df['label']

print("\n--- 2.6. Feature Analysis (Statistical Summary) ---")
# Statistics reveal range, mean, and potential outliers
print(X.describe().T)
print("-" * 30)


# --- 2.7. Feature Insight: Nutrient Relationship (N, P, K) ---
# We look at the mean N, P, K for a few contrasting crops to identify requirements
print("\n--- 2.7. Feature Insight: Contrasting Nutrient Requirements ---")
# High N crop (Rice), High K crop (Banana), Balanced crop (Maize)
contrast_crops = ['rice', 'banana', 'maize', 'coffee']
insight_df = df[df['label'].isin(contrast_crops)].groupby('label')[['N', 'P', 'K', 'ph', 'rainfall']].mean().round(2)
print(insight_df)
print("\n**EDA Insights:**")
print("- **Rice** has the highest mean Rainfall (~294 mm) and N value, confirming it is a high-water, nitrogen-demanding crop.")
print("- **Banana** has the highest mean K value (~51 mg/kg), indicating its potassium demands.")
print("- **Coffee** shows the lowest mean pH (~5.81), highlighting its preference for acidic soil.")
print("-" * 30)

In [ ]:
# Visualize the distribution of Temperature and Humidity
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['temperature'], kde=True, color='purple')
plt.title('Distribution of Temperature')
plt.xlabel('Temperature (°C)')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.histplot(df['humidity'], kde=True, color='teal')
plt.title('Distribution of Humidity')
plt.xlabel('Humidity (%)')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize the distribution of N, P, and K
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.histplot(df['N'], kde=True, color='skyblue')
plt.title('Distribution of Nitrogen (N)')
plt.xlabel('N Value')
plt.ylabel('Count')

plt.subplot(1, 3, 2)
sns.histplot(df['P'], kde=True, color='lightcoral')
plt.title('Distribution of Phosphorus (P)')
plt.xlabel('P Value')
plt.ylabel('Count')

plt.subplot(1, 3, 3)
sns.histplot(df['K'], kde=True, color='lightgreen')
plt.title('Distribution of Potassium (K)')
plt.xlabel('K Value')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# --- 3. Data Preparation (Scaling and Splitting) ---

# Split data
X_full_train, X_test, y_full_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(X_full_train, y_full_train, test_size=.25, random_state=42)

# Scaling features (Essential for distance-based models like KNN and helpful for LR)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("\n✅ Data split (80/20) and features scaled successfully using StandardScaler.")

# print the length for every set of data after splitiing
print(f"X_train: {len(X_train)}, X_test: {len(X_test)}, y_train: {len(y_train)}, y_test: {len(y_test)}, X_val: {len(X_val)}, y_val: {len(y_val)}")


## Model Training and Comparison

### Task:
Train multiple classification models (e.g., Logistic Regression, K-Nearest Neighbors, Random Forest, Decision Tree, Gradient Boosting, XGBoost) and compare their baseline performance.


In [ ]:
print("\n--- 4. Model Training and Comparison ---")


# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000), # Increased max_iter for convergence
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(n_estimators= 100, random_state=42, use_label_encoder=False, eval_metric='logloss')
}

results = {}

# Train and evaluate each model
for name, model in models.items():
  print(f"\nTraining and evaluating {name}...")

  # Train model
  if name == 'XGBoost':
    # XGBoost was trained on X_train (unscaled) for feature importance, so we'll maintain that consistency
    # Encode target labels to numerical values for XGBoost
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)

    model.fit(X_train, y_train_encoded) # Train with unscaled X and encoded y
    y_pred_raw = model.predict(X_test) # Predict with unscaled X_test
    y_pred = le.inverse_transform(y_pred_raw) # Inverse transform predictions to original string labels
  else:
    model.fit(X_train_scaled, y_train) # Train with scaled X and original string y
    y_pred = model.predict(X_test_scaled) # Predict with scaled X_test, predictions will be string labels

  # Evaluate performance
  accuracy = accuracy_score(y_test, y_pred) # y_test is string, y_pred is now also string (for all models)
  report = classification_report(y_test, y_pred)

  results[name] = {'accuracy': accuracy, 'report': report}

  print(f"Accuracy for {name}: {accuracy:.4f}")
  print(f"Classification Report for {name}:\n{report}")
  print("-" * 50)

# Summarize results
print("\n--- Model Comparison Summary ---")
for name, metrics in results.items():
    print(f"{name}: Accuracy = {metrics['accuracy']:.4f}")
print("--------------------------------")

## Hyperparameter Tuning

### Task:
Perform GridSearchCV to optimize hyperparameters for the best performing models.


In [ ]:
print("\n--- 5. Hyperparameter Tuning ---")

# 1. Define parameter grids for the models(Random forest, KNN, Decesion Trees, XGboot, Logestic Regression, Gradient Boosting)
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [10, 20, 30, None],
    'min_samples_leaf': [1, 3, 5, 10],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

param_grid_dt = {
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 3, 5, 10]
}

param_grid_gb = {
    'n_estimators': [50, 100, 150, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 10, None]
}

param_grid_xgb = {
    'n_estimators': [100, 150, 200],
    'learning_rate': [0.05, 0.01, 0.1, 0.3, 0.5, 1],
    'max_depth': [3, 5, 10, None]
}

param_grid_lr = {
  'C': [0.1, 1.0, 10.0]

}

models = [
    { 'classifier': RandomForestClassifier(random_state=42) , 'param_grid': param_grid_rf},
    { 'classifier': KNeighborsClassifier() , 'param_grid': param_grid_knn},
    { 'classifier': DecisionTreeClassifier(random_state=42) , 'param_grid': param_grid_dt},
    { 'classifier': GradientBoostingClassifier(random_state=42) , 'param_grid': param_grid_gb},
    { 'classifier': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss') , 'param_grid': param_grid_xgb},
    { 'classifier': LogisticRegression(random_state=42, max_iter=1000) , 'param_grid': param_grid_lr}
]
print("✅ Parameter grids defined for Random Forest and K-Nearest Neighbors.")

In [ ]:
# Models Hyperparameter Tuning (continued) ---
print("\n--- 5. Hyperparameter Tuning (continued) ---")

# Initialize model_best_features to store results
model_best_features = {

}

for model_info in models:
    model = model_info['classifier']
    param_grid = model_info['param_grid']
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    if model.__class__.__name__ == 'XGBClassifier':
        grid_search.fit(X_train, y_train_encoded)
    else:
      grid_search.fit(X_train_scaled, y_train)

    # Store the best parameters and score for the current model
    model_best_features[model.__class__.__name__] = {
        'best_params': grid_search.best_params_,
        'best_score': grid_search.best_score_,
        # 'feature_importance': grid_search.feature_importances_ if hasattr(model, 'feature_importances_') else None
    }

    print(f"Best parameters for {model.__class__.__name__}: {grid_search.best_params_}")
    print(f"Best accuracy for {model.__class__.__name__}: {grid_search.best_score_}")
    # print feature importance for each model


    print("-" * 50)

print("\n--- Hyperparameter Tuning Results Summary ---")
for name, metrics in model_best_features.items():
    print(f"{name}: Best Accuracy = {metrics['best_score']:.4f}")
    print(f"  Best Params = {metrics['best_params']}")
print("---------------------------------------------")


In [ ]:
# evaluate the models with the tuned parammeters
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, C=10.0),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=3, metric='manhattan', weights='distance'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10, min_samples_leaf=1, min_samples_split=5),
    'Random Forest': RandomForestClassifier(random_state=42, criterion='gini', max_depth=10, max_features='sqrt', min_samples_leaf=1, min_samples_split=10, n_estimators=300),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, learning_rate=0.1, max_depth=3, n_estimators=200),
    'XGBoost': XGBClassifier(n_estimators= 100, max_depth=3, learning_rate=0.3, random_state=42, use_label_encoder=False, eval_metric='logloss')
}

results = {}
print(f"\nTraining and evaluating {name}...")

for name, model in models.items():
  # Train model
  if name == 'XGBoost':
    # XGBoost was trained on X_train (unscaled) for feature importance, so we'll maintain that consistency
    # Encode target labels to numerical values for XGBoost
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)

    model.fit(X_train, y_train_encoded) # Train with unscaled X and encoded y
    y_pred_raw = model.predict(X_test) # Predict with unscaled X_test
    y_pred = le.inverse_transform(y_pred_raw) # Inverse transform predictions to original string labels
  else:
    model.fit(X_train_scaled, y_train) # Train with scaled X and original string y
    y_pred = model.predict(X_test_scaled) # Predict with scaled X_test, predictions will be string labels

  # Evaluate performance
  accuracy = accuracy_score(y_test, y_pred) # y_test is string, y_pred is now also string (for all models)
  report = classification_report(y_test, y_pred)

  results[name] = {'accuracy': accuracy, 'report': report, 'best_estimator_': model.feature_importances_}

  print(f"Accuracy for {name}: {accuracy:.4f}")
  print(f"Classification Report for {name}:\n{report}")
  print("-" * 50)

# Summarize results
print("\n--- Model Comparison Summary ---")
for name, metrics in results.items():
    print(f"{name}: Accuracy = {metrics['accuracy']:.4f}")
print("--------------------------------")


In [ ]:
# evaluate the tuned models and select the best one
best_model = None
best_accuracy = 0.0

# find the best model in the result
for name, metrics in results.items():
    if metrics['accuracy'] > best_accuracy:
        best_accuracy = metrics['accuracy']
        best_model = name

print(f"The best model is {best_model} with an accuracy of {best_accuracy:.4f}")

## Feature Importance Analysis AND Final Evaluation

### Tasks:
Analyze the importance of each feature for the best performing model to understand their contribution to crop prediction.

<br>

Evaluate the best model's performance on the test set using metrics like accuracy and classification report.


In [ ]:
print("\n--- 6. Final Evaluation and Feature Importance ---")

print("\nBest parameters for Random Forest:")

# 1. Get the best estimator from GridSearchCV for Random Forest
best_rf_model = models['Random Forest'].fit(X_train_scaled, y_train)



# 2. Access the feature_importances_ attribute
feature_importances = best_rf_model.feature_importances_

# 3. Create a Pandas Series to store feature names and their importance scores
# 'features' list is available from previous steps
feature_importance_df = pd.Series(feature_importances, index=features)

# 4. Sort the features by importance in descending order
sorted_feature_importance = feature_importance_df.sort_values(ascending=False)

# 5. Print or display the feature importances
print("\nFeature Importances for the Best Random Forest Model:")
print(sorted_feature_importance)
print("----------------------------------------------------")

print("\n--- 6.1. Evaluating the Best Model (Random Forest) ---")


# Make predictions on the scaled test data
y_pred_best_rf = best_rf_model.predict(X_test_scaled)

# Calculate and print accuracy
accuracy_best_rf = accuracy_score(y_test, y_pred_best_rf)
print(f"Overall Accuracy of the Best Random Forest Model: {accuracy_best_rf:.4f}")

# Generate and print a detailed classification report
report_best_rf = classification_report(y_test, y_pred_best_rf)
print(f"\nClassification Report for the Best Random Forest Model:\n{report_best_rf}")
print("----------------------------------------------------")

In [ ]:
# Create a bar plot of feature importances from the best Random Forest model
plt.figure(figsize=(10, 6))
sns.barplot(x=sorted_feature_importance.values, y=sorted_feature_importance.index, palette='viridis')
plt.title('Feature Importances from Best Random Forest Model')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

In [ ]:
top_features = sorted_feature_importance.index.tolist()

# Calculate the correlation matrix for the top features
correlation_matrix = df[top_features].corr()

# Visualize the correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Top Features')
plt.show()

## Model and Scaler Export

### Task:
Save the trained best model and the StandardScaler for future use in predictions.


In [ ]:
print("\n--- 7. Model and Scaler Export ---")

# Save the best Random Forest model
joblib.dump(best_rf_model, MODEL_FILE)
print(f"🎉 SUCCESS: Best model saved to {MODEL_FILE}")

# Save the StandardScaler
joblib.dump(scaler, SCALER_FILE)
print(f"🎉 SUCCESS: Scaler saved to {SCALER_FILE}")